# TAIX-Ray Failure Case Explorer
Randomly samples cases where all 3 linear-probe models (CheXagent, MedSigLIP, BiomedCLIP) failed,
displaying the raw chest X-ray with correct 16-bit → 8-bit normalization.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
import os, random

# The output of analyze_failures.py. Set RADHARMONY_OUTPUT_DIR to its output folder.
# The default '../outputs' is model/outputs when Jupyter starts in model/notebooks.
DATA_DIR = os.path.join(os.environ.get('RADHARMONY_OUTPUT_DIR') or '../outputs', 'taix_failure_analysis')
LABELS = [
    'atelectasis_left', 'atelectasis_right', 'heart_size',
    'pleural_effusion_left', 'pleural_effusion_right',
    'pulmonary_congestion', 'pulmonary_opacities_left', 'pulmonary_opacities_right',
]
BACKBONES = ['chexagent', 'medsiglip', 'biomedclip']

df = pd.read_parquet(f'{DATA_DIR}/taix_lp_predictions.parquet')
print(f'{len(df):,} test samples | {df["any_label_all_fail"].sum():,} have ≥1 all-fail label')

In [ ]:
def load_cxr(path):
    """Load 16-bit PNG and normalise to displayable uint8."""
    arr = np.array(Image.open(path)).astype(np.float32)
    lo, hi = np.percentile(arr, 1), np.percentile(arr, 99)
    arr = np.clip((arr - lo) / (hi - lo + 1e-6), 0, 1)
    return (arr * 255).astype(np.uint8)

def pred_summary(row, lbl):
    true = int(row[f'true_{lbl}'])
    lines = [f"True: {'POS' if true else 'neg'}"]
    for b in BACKBONES:
        p = row[f'prob_{b}_{lbl}']
        pred = int(row[f'pred_{b}_{lbl}'])
        err = bool(row[f'err_{b}_{lbl}'])
        mark = '✗' if err else '✓'
        lines.append(f"{mark} {b[:5]}: {'pos' if pred else 'neg'} ({p:.2f})")
    return '\n'.join(lines)

## 1 · Random sample of all-model-failure cases
Change `FOCUS_LABEL` or `N` to explore.

In [ ]:
FOCUS_LABEL = 'pulmonary_congestion'   # ← change this
N = 12                                  # ← images per grid
SEED = 42

subset = df[df[f'all_fail_{FOCUS_LABEL}'] == 1].sample(n=min(N, len(df)), random_state=SEED)
print(f'{len(subset)} cases shown  |  focus: {FOCUS_LABEL}')

ncols = 4
nrows = (N + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 4.5))
axes = axes.flatten()

for i, (_, row) in enumerate(subset.iterrows()):
    ax = axes[i]
    try:
        img = load_cxr(row['image_path'])
        ax.imshow(img, cmap='gray', vmin=0, vmax=255)
    except Exception as e:
        ax.text(0.5, 0.5, f'Load error:\n{e}', transform=ax.transAxes, ha='center')
    
    true_val = int(row[f'true_{FOCUS_LABEL}'])
    n_fail = int(row['n_labels_all_fail'])
    ax.set_title(
        f"{row['study_id'][:20]}\n"
        f"Age {int(row['age'])//365}y {row['sex']}  |  {n_fail}/8 labels fail\n"
        f"True: {'POS' if true_val else 'neg'}",
        fontsize=7.5, pad=3
    )
    # Annotation strip
    summary = pred_summary(row, FOCUS_LABEL)
    ax.text(0.01, 0.01, summary, transform=ax.transAxes,
            fontsize=6.5, color='yellow', va='bottom',
            bbox=dict(facecolor='black', alpha=0.55, pad=2))
    ax.axis('off')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle(f'All-model failures · {FOCUS_LABEL}', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 2 · Full prediction table for a single study
Paste a `study_id` from the grid above to drill into a specific case.

In [ ]:
STUDY_ID = subset.iloc[0]['study_id']   # ← paste any full study_id here

row = df[df['study_id'] == STUDY_ID].iloc[0]
print('study_id :', row['study_id'])
print('image    :', row['image_path'])
print(f'Age {int(row["age"])//365}y  Sex {row["sex"]}  Date {row["study_date"]}')
print(f'Labels failing all models: {int(row["n_labels_all_fail"])}/8')
print()

records = []
for lbl in LABELS:
    true_val = int(row[f'true_{lbl}'])
    r = {'label': lbl, 'true': 'POS' if true_val else 'neg', 'all_fail': bool(row[f'all_fail_{lbl}'])}
    for b in BACKBONES:
        r[f'{b}_prob'] = round(float(row[f'prob_{b}_{lbl}']), 3)
        r[f'{b}_pred'] = 'pos' if int(row[f'pred_{b}_{lbl}']) else 'neg'
        r[f'{b}_err']  = '✗' if bool(row[f'err_{b}_{lbl}']) else '✓'
    records.append(r)

display(pd.DataFrame(records))

## 3 · Compare a single image: raw vs contrast-stretched

In [ ]:
raw = np.array(Image.open(row['image_path']))
normalised = load_cxr(row['image_path'])

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

axes[0].imshow(raw, cmap='gray')
axes[0].set_title(f'Raw (uint16 0–{raw.max()})')

axes[1].imshow(normalised, cmap='gray', vmin=0, vmax=255)
axes[1].set_title('1–99th percentile stretch')

# CLAHE-like: histogram equalization
from PIL import ImageOps
eq = np.array(ImageOps.equalize(Image.fromarray(normalised)))
axes[2].imshow(eq, cmap='gray')
axes[2].set_title('Histogram equalised')

for ax in axes: ax.axis('off')
fig.suptitle(row['study_id'], fontsize=9)
plt.tight_layout()
plt.show()

print(f'Raw pixel stats:  min={raw.min()}  max={raw.max()}  mean={raw.mean():.0f}  std={raw.std():.0f}')

## 4 · FP vs FN breakdown per label

In [ ]:
rows = []
for lbl in LABELS:
    fail = df[df[f'all_fail_{lbl}'] == 1]
    fp = (fail[f'true_{lbl}'] == 0).sum()
    fn = (fail[f'true_{lbl}'] == 1).sum()
    rows.append({'label': lbl, 'all_fail_N': len(fail), 'FP (false alarm)': fp, 'FN (missed)': fn,
                 'FP%': f'{fp/len(fail)*100:.1f}%' if len(fail) else '-',
                 'FN%': f'{fn/len(fail)*100:.1f}%' if len(fail) else '-'})

summary = pd.DataFrame(rows).sort_values('all_fail_N', ascending=False)
display(summary)